# 04 — TVG Construction

**Step A (one-time):** fetch buildings + street network across the study
area's bounding box, precompute everything that doesn't depend on any
individual incident (building shape metrics, building-type vocab,
highway-type vocab, betweenness centrality, orientation entropy), build
the shared STRtree, cache to disk.

**Step B (per point, checkpointed):** road-projection, isovist ray-casting,
node/edge construction, save graph + QC map.

Uses `src/geo_utils.py`, `src/osm_fetch.py`, `src/isovist.py`,
`src/tvg_builder.py`, `src/tvg_visualize.py`, `src/manifest.py`.
CPU is sufficient.

In [ ]:
# ── Clone/update repo, mount Drive ──────────────────────────────────────
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q osmnx geopandas shapely pyproj networkx torch_geometric tqdm pyyaml pandas numpy matplotlib

In [ ]:
import yaml
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/tvg_schema.yaml") as f:
    tvg_cfg = yaml.safe_load(f)

INTERIM_DIR = Path(paths_cfg["interim_dir"])
PROCESSED_DIR = Path(paths_cfg["processed_dir"])

OSM_CACHE_DIR = INTERIM_DIR / "osm_cache"
TVG_OUT_DIR = PROCESSED_DIR / "tvg_graphs"
VIZ_OUT_DIR = INTERIM_DIR / "tvg_visualizations"
LOG_PATH = INTERIM_DIR / "tvg_construction_log.csv"

TVG_OUT_DIR.mkdir(parents=True, exist_ok=True)
VIZ_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Isovist radius: {tvg_cfg['isovist_radius_m']}m | crash_history: {tvg_cfg['crash_history_threshold_m']}m | N_RAYS: {tvg_cfg['n_rays']}")

In [ ]:
import manifest
import geo_utils
import osm_fetch
import isovist as iso
import tvg_builder
import tvg_visualize

In [ ]:
# ── STEP A: one-time fetch + precompute (skipped entirely if cache exists) ──
from tqdm.auto import tqdm

if osm_fetch.cache_exists(OSM_CACHE_DIR):
    print("✅ OSM cache already exists — loading, skipping re-fetch/re-compute.")
    (buildings_gdf, building_type_vocab, highway_vocab, G,
     betweenness, orientation_entropy, utm_crs) = osm_fetch.load_cache(OSM_CACHE_DIR)
else:
    print("No cache found — fetching and precomputing (runs once)...")

    with tqdm(total=6, desc="Step A") as pbar:
        buildings_gdf, G, utm_crs = osm_fetch.fetch_study_area_layers(
            paths_cfg["boundary_geojson"], tvg_cfg["bbox_padding_m"], tvg_cfg["network_type"]
        )
        pbar.set_description("Fetched buildings + streets"); pbar.update(1)

        buildings_gdf = osm_fetch.compute_building_shape_metrics(buildings_gdf)
        pbar.set_description("Computed building shape metrics"); pbar.update(1)

        buildings_gdf, building_type_vocab = osm_fetch.build_building_type_vocab(buildings_gdf)
        pbar.set_description("Built building-type vocab"); pbar.update(1)

        highway_vocab = osm_fetch.build_highway_vocab(G)
        pbar.set_description("Built highway-type vocab"); pbar.update(1)

        betweenness, orientation_entropy = osm_fetch.compute_intersection_metrics(G)
        pbar.set_description("Computed betweenness + orientation entropy"); pbar.update(1)

        osm_fetch.save_cache(OSM_CACHE_DIR, buildings_gdf, building_type_vocab, highway_vocab,
                              G, betweenness, orientation_entropy, utm_crs)
        pbar.set_description("Cached to disk"); pbar.update(1)

print(f"\nBuildings: {len(buildings_gdf)} | Street nodes: {len(G.nodes)} | "
      f"Building types: {len(building_type_vocab)} | Highway types: {len(highway_vocab)}")

In [ ]:
# ── Build the shared STRtree (kept in memory for the whole run — not cached
#    to disk, since STRtree objects aren't easily picklable; cheap to rebuild
#    once per session from the cached buildings layer). ─────────────────
tree, boundaries = osm_fetch.build_building_strtree(buildings_gdf)
buildings_sindex = buildings_gdf.sindex  # for tvg_visualize.py's fast window queries
print(f"STRtree built over {len(boundaries)} building boundaries.")

In [ ]:
# ── Load reconciled points from 01, project all to UTM once (needed for
#    fast peer-incident distance lookups without re-projecting per call) ──
import pandas as pd
import geopandas as gpd

reconciled = pd.read_parquet(INTERIM_DIR / "reconciled_points.parquet")
utm_points = gpd.GeoDataFrame(
    reconciled, geometry=gpd.points_from_xy(reconciled["input_lon"], reconciled["input_lat"]), crs="EPSG:4326"
).to_crs(utm_crs)
reconciled["_utm_x"] = utm_points.geometry.x
reconciled["_utm_y"] = utm_points.geometry.y

all_point_ids = reconciled["point_id"].tolist()
pending = manifest.pending_items(all_point_ids, TVG_OUT_DIR, ext=".pt")
print(f"Total points: {len(all_point_ids)}  |  Already done: {len(all_point_ids) - len(pending)}  |  Pending: {len(pending)}")

In [ ]:
# ── STEP B: main per-point loop — checkpointed, resumable ───────────────
import torch

row_lookup = reconciled.set_index("point_id", drop=False)

for point_id in tqdm(pending, desc="Building TVG"):
    try:
        incident_row = row_lookup.loc[point_id]

        data, meta = tvg_builder.process_incident(
            incident_row, reconciled, buildings_gdf, building_type_vocab, highway_vocab, G,
            betweenness, orientation_entropy, tree, boundaries, utm_crs,
            isovist_radius_m=tvg_cfg["isovist_radius_m"],
            n_rays=tvg_cfg["n_rays"],
            crash_history_threshold_m=tvg_cfg["crash_history_threshold_m"],
        )

        torch.save(data, TVG_OUT_DIR / f"{point_id}.pt")

        peer_xy = [
            (row_lookup.loc[pid, "_utm_x"], row_lookup.loc[pid, "_utm_y"])
            for pid in meta["peer_point_ids"]
        ]
        fig = tvg_visualize.render_tvg_overlay(
            meta["origin_xy"], int(incident_row["label"]), meta["polygon"],
            meta["included_building_ids"], buildings_gdf, buildings_sindex, peer_xy,
            G=G, data=data, included_intersections=meta["included_intersections"],
            u=meta["u"], v=meta["v"],
        )
        tvg_visualize.save_overlay(fig, VIZ_OUT_DIR, point_id)

        manifest.append_log(LOG_PATH, point_id, "tvg_construction", "ok")

    except Exception as e:
        manifest.append_log(LOG_PATH, point_id, "tvg_construction", "error", str(e))
        tqdm.write(f"  ⚠️  {point_id}: {e}")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────
log = manifest.load_log(LOG_PATH)
n_remaining = len(manifest.pending_items(all_point_ids, TVG_OUT_DIR, ext=".pt"))
print(f"Remaining pending after this run: {n_remaining} / {len(all_point_ids)}")

if "status" in log.columns and (log["status"] == "error").any():
    errors = log[log["status"] == "error"]
    print(f"\n⚠️  {len(errors)} points failed — re-running this notebook will retry them.")
    display(errors[["point_id", "error", "timestamp"]].tail(20))
else:
    print("\n✅ No errors logged.")

In [ ]:
# ── QC: node-count distributions ─────────────────────────────────────────
import seaborn as sns
import matplotlib.pyplot as plt

counts = []
sample_ids = [pid for pid in all_point_ids if manifest.is_done(TVG_OUT_DIR, pid, ext=".pt")]
for pid in tqdm(sample_ids, desc="Scanning graph sizes"):
    d = torch.load(TVG_OUT_DIR / f"{pid}.pt", weights_only=False)
    counts.append({
        "point_id": pid,
        "n_buildings": d["building"].x.shape[0],
        "n_intersections": d["intersection"].x.shape[0],
        "n_peers": d["peer_incident"].x.shape[0],
    })

counts_df = pd.DataFrame(counts)
print(counts_df.describe())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["n_buildings", "n_intersections", "n_peers"]):
    sns.histplot(counts_df[col], bins=20, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig(INTERIM_DIR / "qc_tvg_node_count_distribution.png", dpi=150)
plt.show()

n_zero_buildings = (counts_df["n_buildings"] == 0).sum()
if n_zero_buildings:
    print(f"\n⚠️  {n_zero_buildings} points have ZERO included buildings — expected given "
          f"the 50m isovist radius in lower-density areas, but worth spot-checking "
          f"{VIZ_OUT_DIR} if this is a large fraction of the dataset.")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set Seaborn theme
sns.set_theme(style="whitegrid", palette="pastel")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

plot_configs = [
    ("n_buildings", "Buildings per point"),
    ("n_intersections", "Intersections per point"),
    ("n_peers", "Peer incidents per point"),
]

for ax, (col, xlabel) in zip(axes, plot_configs):
    sns.histplot(
        data=counts_df,
        x=col,
        bins=30,
        kde=True,
        color="grey",
        ax=ax
    )

    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel("Count", fontsize=10)
    ax.set_title("")

plt.tight_layout()
plt.savefig(INTERIM_DIR / "qc_tvg_node_count_distribution.png", dpi=150)
plt.show()

n_zero_buildings = (counts_df["n_buildings"] == 0).sum()
if n_zero_buildings:
    print(
        f"\n⚠️ {n_zero_buildings} points have ZERO included buildings — "
        f"expected given the 50 m isovist radius in lower-density areas, "
        f"but worth spot-checking {VIZ_OUT_DIR} if this is a large fraction of the dataset."
    )

In [ ]:
# ── QC: node-count distributions (positive / negative only) ─────────────
import seaborn as sns
import matplotlib.pyplot as plt

def scan_node_counts(point_ids, desc="Scanning graph sizes"):
    counts = []
    for pid in tqdm(point_ids, desc=desc):
        d = torch.load(SVG_OUT_DIR / f"{pid}.pt", weights_only=False)
        counts.append({
            "point_id": pid,
            "n_signage": d["signage"].x.shape[0],
            "n_light_pole": d["light_pole"].x.shape[0],
            "n_road_marking": d["road_marking"].x.shape[0],
            "n_building": d["building"].x.shape[0],
            "n_vegetation": d["vegetation"].x.shape[0],
        })
    return pd.DataFrame(counts)


def plot_node_count_distribution(counts_df, label, color, out_path):
    print(f"\n=== {label} (n={len(counts_df)}) ===")
    print(counts_df.describe())

    cols = ["n_signage", "n_light_pole", "n_road_marking", "n_building", "n_vegetation"]
    fig, axes = plt.subplots(1, len(cols), figsize=(5 * len(cols), 4))
    for ax, col in zip(axes, cols):
        sns.histplot(counts_df[col], kde=True, bins=20, color=color, ax=ax)
        ax.set_title(col)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()

    n_zero = (counts_df[cols].sum(axis=1) == 0).sum()
    if n_zero:
        print(f"\n⚠️  [{label}] {n_zero} points have ZERO detected objects across all types — "
              f"expected occasionally, but worth spot-checking "
              f"{VIZ_OUT_DIR} if this is a large fraction of the dataset.")


# All done points, split by prefix
sample_ids = [pid for pid in all_point_ids if manifest.is_done(SVG_OUT_DIR, pid, ext=".pt")]
positive_ids = [pid for pid in sample_ids if pid.startswith("positive_")]
negative_ids = [pid for pid in sample_ids if pid.startswith("negative_")]

print(f"Total: {len(sample_ids)} | Positive: {len(positive_ids)} | Negative: {len(negative_ids)}")

# --- Positive only (red) ---
counts_df_pos = scan_node_counts(positive_ids, desc="Scanning graph sizes (positive)")
plot_node_count_distribution(
    counts_df_pos, "positive", "red", INTERIM_DIR / "qc_svg_node_count_distribution_positive.png"
)

# --- Negative only (blue) ---
counts_df_neg = scan_node_counts(negative_ids, desc="Scanning graph sizes (negative)")
plot_node_count_distribution(
    counts_df_neg, "negative", "blue", INTERIM_DIR / "qc_svg_node_count_distribution_negative.png"
)

In [ ]:
# ── QC: node-count distributions (positive / negative only) ─────────────
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette="pastel")

NODE_TYPES = {
    "n_buildings": ("building", "Buildings per point"),
    "n_intersections": ("intersection", "Intersections per point"),
    "n_peers": ("peer_incident", "Peer incidents per point"),
}


def scan_node_counts(point_ids, desc="Scanning graph sizes"):
    counts = []
    for pid in tqdm(point_ids, desc=desc):
        data = torch.load(TVG_OUT_DIR / f"{pid}.pt", weights_only=False)

        counts.append({
            "point_id": pid,
            "n_buildings": data["building"].x.shape[0],
            "n_intersections": data["intersection"].x.shape[0],
            "n_peers": data["peer_incident"].x.shape[0],
        })

    return pd.DataFrame(counts)


def plot_node_count_distribution(counts_df, label, color, out_path):
    print(f"\n=== {label} (n={len(counts_df)}) ===")
    print(counts_df.describe())

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    for ax, (col, (_, xlabel)) in zip(axes, NODE_TYPES.items()):
        sns.histplot(
            data=counts_df,
            x=col,
            bins=30,
            kde=True,
            color=color,
            ax=ax,
        )

        ax.set_xlabel(xlabel, fontsize=10)
        ax.set_ylabel("Count", fontsize=10)
        ax.set_title("")

    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()

    n_zero = (counts_df[list(NODE_TYPES.keys())].sum(axis=1) == 0).sum()
    if n_zero:
        print(
            f"\n⚠️ [{label}] {n_zero} points have ZERO nodes across all TVG node types — "
            f"expected occasionally, but worth spot-checking their visualizations "
            f"in {VIZ_OUT_DIR} if this is a large fraction of the dataset."
        )


# All done points, split by prefix
sample_ids = [pid for pid in all_point_ids if manifest.is_done(TVG_OUT_DIR, pid, ext=".pt")]
positive_ids = [pid for pid in sample_ids if pid.startswith("positive_")]
negative_ids = [pid for pid in sample_ids if pid.startswith("negative_")]

print(f"Total: {len(sample_ids)} | Positive: {len(positive_ids)} | Negative: {len(negative_ids)}")

# --- Positive only (red) ---
counts_df_pos = scan_node_counts(positive_ids, desc="Scanning graph sizes (positive)")
plot_node_count_distribution(
    counts_df_pos,
    "positive",
    "red",
    INTERIM_DIR / "qc_tvg_node_count_distribution_positive.png",
)

# --- Negative only (blue) ---
counts_df_neg = scan_node_counts(negative_ids, desc="Scanning graph sizes (negative)")
plot_node_count_distribution(
    counts_df_neg,
    "negative",
    "blue",
    INTERIM_DIR / "qc_tvg_node_count_distribution_negative.png",
)

In [ ]:
print(f"TVG graphs saved to: {TVG_OUT_DIR}")
print(f"QC maps saved to: {VIZ_OUT_DIR}")
print()
print("Next: 05_dataset_assembly.ipynb")